In [2]:
# Importazione librerie necessarie
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Impostiamo lo stile dei grafici per essere professionali
sns.set_theme(style="whitegrid")

# --- 1. CARICAMENTO DEI DATASET ---
print("--- CARICAMENTO DATI ---")
try:
    customers_df = pd.read_csv('customers.csv')
    orders_df = pd.read_csv('orders.csv')
    print("✅ File caricati con successo!")
except FileNotFoundError:
    print("❌ Errore: Assicurati che i file CSV siano nella stessa cartella del notebook.")

# --- 2. AUDIT STRUTTURALE ---
print("\n--- AUDIT: CUSTOMERS ---")
display(customers_df.head(3))
print(customers_df.info())

print("\n--- AUDIT: ORDERS ---")
display(orders_df.head(3))
print(orders_df.info())

# --- 3. CHECK QUALITÀ DATI (I 3 Pilastri) ---
print("\n--- QUALITÀ DATI (Valori Mancanti e Duplicati) ---")
print(f"Valori mancanti in Customers:\n{customers_df.isnull().sum()}\n")
print(f"Valori mancanti in Orders:\n{orders_df.isnull().sum()}\n")

print(f"Duplicati in Customers: {customers_df.duplicated().sum()}")
print(f"Duplicati in Orders: {orders_df.duplicated().sum()}")

--- CARICAMENTO DATI ---
✅ File caricati con successo!

--- AUDIT: CUSTOMERS ---


,Customer_ID,Age,Gender,Privacy,tenure,Region
0,A89996572,78,F,Y,14,Marche
1,A89996584,76,F,Y,14,Lombardia
2,A89996591,77,F,N,14,Puglia


<class 'pandas.DataFrame'>
RangeIndex: 65853 entries, 0 to 65852
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   Customer_ID  65853 non-null  str  
 1   Age          65853 non-null  int64
 2   Gender       65853 non-null  str  
 3   Privacy      64097 non-null  str  
 4   tenure       65853 non-null  str  
 5   Region       64172 non-null  str  
dtypes: int64(1), str(5)
memory usage: 3.0 MB
None

--- AUDIT: ORDERS ---


,year,Customer_ID,Order_Nr,Product_category,Quantity,Value
0,2024,A89996572,I133144425,Beauty,2,88.00211
1,2024,A89996572,I133155840,Beauty,3,233.65079
2,2024,A89996572,I133223693,Beauty,1,71.01459


<class 'pandas.DataFrame'>
RangeIndex: 146415 entries, 0 to 146414
Data columns (total 6 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   year              146415 non-null  int64  
 1   Customer_ID       146415 non-null  str    
 2   Order_Nr          146415 non-null  str    
 3   Product_category  146415 non-null  str    
 4   Quantity          146415 non-null  int64  
 5   Value             146415 non-null  float64
dtypes: float64(1), int64(2), str(3)
memory usage: 6.7 MB
None

--- QUALITÀ DATI (Valori Mancanti e Duplicati) ---
Valori mancanti in Customers:
Customer_ID       0
Age               0
Gender            0
Privacy        1756
tenure            0
Region         1681
dtype: int64

Valori mancanti in Orders:
year                0
Customer_ID         0
Order_Nr            0
Product_category    0
Quantity            0
Value               0
dtype: int64

Duplicati in Customers: 0
Duplicati in Orders: 0


In [3]:
# --- FASE 2: PULIZIA DEI DATI (Data Quality) ---
print("--- INIZIO PULIZIA E GESTIONE OUTLIER ---")

# Creiamo le copie per la pulizia
customers = customers_df.copy()
orders = orders_df.copy()

# 1. Gestione Valori Mancanti (Come da soluzione proposta)
customers['Privacy'] = customers['Privacy'].fillna('N')
customers['Region'] = customers['Region'].fillna('Unknown')
# (Opzionale: gestiamo anche eventuali NaN in tenure se presenti in altri sample)
if customers['tenure'].isnull().sum() > 0:
    customers['tenure'] = customers['tenure'].fillna(customers['tenure'].median())

# 2. Gestione Outlier (Capping al 99° percentile)
# Cap Età a 90 anni
customers['Age'] = customers['Age'].clip(upper=90)

# Cap Quantità Ordini estremi
q_quant = orders['Quantity'].quantile(0.99)
orders['Quantity'] = orders['Quantity'].clip(upper=q_quant)

# Cap Valore Ordini per Categoria Prodotto (Evita che transazioni assurde rovinino l'AOV)
def cap_by_category(x):
    return x.clip(upper=x.quantile(0.99))
orders['Value'] = orders.groupby('Product_category')['Value'].transform(cap_by_category)

print("✅ Pulizia completata!")
print(f"Valori mancanti residui in Customers: {customers.isnull().sum().sum()}")
print(f"Valori mancanti residui in Orders: {orders.isnull().sum().sum()}")

# --- 3. UNIONE DEI DATASET (Merge) ---
print("\n--- MERGE DEI DATASET ---")
# Uniamo gli ordini con i dettagli dei clienti tramite 'Customer_ID'
merged_df = pd.merge(orders, customers, on='Customer_ID', how='left')

# Convertiamo l'anno in stringa/categoria per evitare che venga trattato come un numero con decimali
if 'year' in merged_df.columns:
    merged_df['year'] = merged_df['year'].astype(int).astype('category')

print(f"Dimensioni del Dataset Finale (Merged): {merged_df.shape}")
display(merged_df.head())

--- INIZIO PULIZIA E GESTIONE OUTLIER ---
✅ Pulizia completata!
Valori mancanti residui in Customers: 0
Valori mancanti residui in Orders: 0

--- MERGE DEI DATASET ---
Dimensioni del Dataset Finale (Merged): (146415, 11)


,year,Customer_ID,Order_Nr,Product_category,Quantity,Value,Age,Gender,Privacy,tenure,Region
0,2024,A89996572,I133144425,Beauty,2,88.00211,78,F,Y,14,Marche
1,2024,A89996572,I133155840,Beauty,3,233.65079,78,F,Y,14,Marche
2,2024,A89996572,I133223693,Beauty,1,71.01459,78,F,Y,14,Marche
3,2024,A89996572,I133249311,Beauty,3,102.43185,78,F,Y,14,Marche
4,2024,A89996605,I133249113,Beauty,4,310.19115,76,F,Y,14,Emilia-Romagna


In [4]:
# --- FASE 3: CUSTOMER TRENDS & KPIs ---
print("--- CALCOLO DEI KPI ANNUALI ---")

# Assicuriamoci che l'anno sia in formato numerico per poterlo ordinare e analizzare
merged_df['year_int'] = merged_df['year'].astype(int)

# 1. Aggregazioni base raggruppate per anno
summary = merged_df.groupby('year_int', sort=True).agg(
    Total_Revenue = ('Value', 'sum'),
    Active_Customers = ('Customer_ID', 'nunique'),
    Total_Orders = ('Order_Nr', 'nunique'),
    Total_Quantity = ('Quantity', 'sum')
).reset_index().rename(columns={'year_int': 'year'})

# 2. Calcolo dei KPI Derivati (come spiegato nelle slide dell'EDA)
summary['Order_Frequency'] = summary['Total_Orders'] / summary['Active_Customers']
summary['AOV'] = summary['Total_Revenue'] / summary['Total_Orders']
summary['Avg_Spending'] = summary['Total_Revenue'] / summary['Active_Customers']
summary['Units_per_Customer'] = summary['Total_Quantity'] / summary['Active_Customers']
summary['Avg_Price'] = summary['Total_Revenue'] / summary['Total_Quantity']

# 3. Calcolo del Retention Rate (Tasso di Fedeltà)
# Estraiamo gli ID univoci dei clienti per ogni anno
year_customers = merged_df.groupby('year_int')['Customer_ID'].apply(set).to_dict()
sorted_years = sorted(year_customers.keys())

retention_rates = {}
for i, yr in enumerate(sorted_years):
    if i == 0:
        # Il primo anno non ha uno storico precedente
        retention_rates[yr] = float('nan') 
    else:
        # Confrontiamo i clienti dell'anno attuale con quelli dell'anno prima
        prev_yr = sorted_years[i - 1]
        prev_set = year_customers[prev_yr]
        curr_set = year_customers[yr]
        
        retained = len(prev_set & curr_set)
        retention_rates[yr] = round(retained / len(prev_set) * 100, 2)

# Aggiungiamo la colonna alla tabella
summary['Retention_Rate_pct'] = summary['year'].map(retention_rates)

# Mostriamo la tabella formattata per una facile lettura (2 decimali)
pd.set_option('display.float_format', '{:,.2f}'.format)
display(summary.set_index('year'))

--- CALCOLO DEI KPI ANNUALI ---


,Total_Revenue,Active_Customers,Total_Orders,Total_Quantity,Order_Frequency,AOV,Avg_Spending,Units_per_Customer,Avg_Price,Retention_Rate_pct
year,,,,,,,,,,
2024,"7,710,026.38",33940,50630,92373,1.49,152.28,227.17,2.72,83.47,NaN
2025,"7,353,523.37",32745,50728,87764,1.55,144.96,224.57,2.68,83.79,39.73
2026,"6,317,162.45",29204,44832,75775,1.54,140.91,216.31,2.59,83.37,39.00


# Executive Summary: Analisi Calo Vendite (2024-2026)

## 1. Il Trend del Calo
I ricavi totali hanno subito una contrazione di circa 1,4 milioni di euro, passando da 7,71M€ nel 2024 a 6,31M€ nel 2026. Questo calo è guidato da due fattori concomitanti: la perdita di utenti attivi (scesi da 33.940 a 29.204) e una lieve ma costante flessione dello scontrino medio (AOV), sceso da 152€ a 140€.

## 2. Il Problema della Retention (Fidelizzazione)
L'analisi evidenzia una criticità strutturale: il **Retention Rate è bloccato al 39%**. Ciò significa che il nostro e-commerce perde circa il 60% della sua base clienti ogni singolo anno. Non stiamo riuscendo a trasformare i nuovi acquirenti in clienti abituali di lungo termine.

## 3. Actionable Recommendations (Piano d'Azione)
* **Strategie di Win-Back (Re-engagement):** Dato l'elevato tasso di abbandono (60%), è prioritario attivare campagne di email marketing automatizzate (con sconti mirati) per recuperare i clienti inattivi dell'anno precedente prima che li acquisisca la concorrenza.
* **Incentivi all'Up-Selling per alzare l'AOV:** Per contrastare il calo dello scontrino medio da 152€ a 140€, suggeriamo l'introduzione di bundle di prodotti ("Acquista insieme e risparmia") o soglie di spedizione gratuita dinamiche impostate a 160€ per spingere l'utente a inserire un articolo extra nel carrello.
* **Analisi della Customer Satisfaction:** Un tasso di mancato ritorno così alto fisiologicamente suggerisce un'esperienza post-vendita migliorabile (es. tempi di spedizione, qualità o customer care). Consigliamo di inviare sondaggi mirati ai clienti del 2025 che non hanno riacquistato nel 2026.